# Summarise and plot feature coefficients across cross-validation folds

**Notebook:** `07_analyze_coefficients.ipynb`

## Purpose

- Reads the coefficient files written by scripts 01-06 (Cox: `*_HazardRatios_ByFold.csv`; DeepSurv: `*_FeatureCoefficients.csv`).
- Aggregates each feature's coefficient across the 10 folds on the log scale, with a Nadeau-Bengio corrected 95% confidence interval, and exponentiates to hazard ratios.
- Plots the top features for Cox vs DeepSurv for each feature-selection method.

## Inputs (read from `INPUT_DIR`)

- `cox_feature_subset_HazardRatios_ByFold.csv`
- `cox_powershap_HazardRatios_ByFold.csv`
- `cox_shap_filter_HazardRatios_ByFold.csv`
- `deepsurv_feature_subset_FeatureCoefficients.csv`
- `deepsurv_powershap_FeatureCoefficients.csv`
- `deepsurv_shap_filter_FeatureCoefficients.csv`

## Outputs (written to the figures folder)

- `feature_subset.png`
- `powershap.png`
- `shap_filter.png`
- `cox.png`
- `deepsurv.png`

## Configuration

| Variable | Value in this notebook | Meaning |
|---|---|---|
| `INPUT_DIR` | `"output/"` | Folder holding the CSVs from scripts 01-06 (must end with `/`) |
| `OUTPUT_DIR` | `"figures"` | Folder where figures are saved (must exist) |

## Requirements

Python packages imported by this notebook: `matplotlib`, `numpy`, `pandas`, `scipy`.


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Folder containing the CSV files written by scripts 01-06 (must end with "/")
INPUT_DIR = "output/"
# Folder where figures are saved (must exist before running)
OUTPUT_DIR = "figures"

feature_subset_deepsurv = pd.read_csv(f"{INPUT_DIR}deepsurv_feature_subset_FeatureCoefficients.csv")
powershap_deepsurv = pd.read_csv(f"{INPUT_DIR}deepsurv_powershap_FeatureCoefficients.csv")
shap_filter_deepsurv = pd.read_csv(f"{INPUT_DIR}deepsurv_shap_filter_FeatureCoefficients.csv")

feature_subset_cox = pd.read_csv(f"{INPUT_DIR}cox_feature_subset_HazardRatios_ByFold.csv")
powershap_cox = pd.read_csv(f"{INPUT_DIR}cox_powershap_HazardRatios_ByFold.csv")
shap_filter_cox = pd.read_csv(f"{INPUT_DIR}cox_shap_filter_HazardRatios_ByFold.csv")

In [ ]:
import pandas as pd
 
# Create the variable label lookup table
var_labels = pd.DataFrame({
    "feature": [
        "age_cat_btwn40to49", "age_cat_btwn50to59", "age_cat_btwn60to69", "age_cat_gte70",
 
        "cd4_2", "cd4_3",
 
        "cd4_ini_2", "cd4_ini_3",
 
        "art_ini_2", "art_ini_3",
 
        "riskgroup_Unknown", "riskgroup_MSM",
 
        "efavirenz_1", "dolutegravir_1", "emtricitabine_1",
        "ritonavir_1", "cobicistat_1", "tenofovir_taf_1",
        "tenofovir_1",
 
        "tc_lipidmed_2", "anxiety_1", "smoking_1",
 
        "race_2", "race_4",
 
        "hemoglobin"
    ],
 
    "label": [
        "Age 40-49",
        "Age 50-59",
        "Age 60-69",
        "Age 70+",
 
        "CD4: 250-499",
        "CD4: 500+",
 
        "CD4 at ART initiation: 250-499",
        "CD4 at ART initiation: 500+",
 
        "NNRTI regimen at ART initiation",
        "INSTI regimen at ART initiation",
 
        "Unknown HIV acquisition risk",
        "MSM HIV acquisition risk",
 
        "Efavirenz",
        "Dolutegravir",
        "Emtricitabine",
        "Ritonavir",
        "Cobicistat",
        "Tenofovir TAF",
        "Tenofovir",
 
        "Lipid-lowering medication use",
        "Anxiety",
        "Ever smoking",
 
        "Non-Hispanic Black race",
        "Other or unknown race/ethnicity",
 
        "Hemoglobin"
    ]
})
 
# Function to add labels
def add_labels(final_summary, var_labels=var_labels):
    """
    Left-join variable labels onto a summary DataFrame.
 
    Parameters
    ----------
    final_summary : pandas.DataFrame
        Must contain a 'feature' column.
    var_labels : pandas.DataFrame
        Lookup table with 'feature' and 'label' columns.
 
    Returns
    -------
    pandas.DataFrame
    """
    return final_summary.merge(var_labels, on="feature", how="left")

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import t as t_dist

# 1. Define the Nadeau-Bengio correction for 10-fold CV
def calculate_nadeau_bengio_bounds(group_data, k=10, confidence=0.95):
    # Determine the correct log-scale column based on the model type
    if 'coefficient' in group_data.columns:
        coefs = group_data['coefficient'].values
    elif 'log_coef' in group_data.columns:
        coefs = group_data['log_coef'].values
    else:
        # Fallback if the dataframe only contains the exponentiated ratios
        ratio_col = 'risk_ratio' if 'risk_ratio' in group_data.columns else 'hazard_ratio'
        coefs = np.log(group_data[ratio_col].values)
    
    # Calculate mean and variance on the LOG scale
    mean_coef = np.mean(coefs)
    s2 = np.var(coefs, ddof=1)
    
    # Nadeau-Bengio corrected variance and standard error
    corrected_var = (1 / k + 1 / (k - 1)) * s2
    corrected_se = np.sqrt(corrected_var)
    
    # Calculate margin of error
    t_crit = t_dist.ppf((1 + confidence) / 2, df=k - 1)
    margin = t_crit * corrected_se
    
    lower_bound_log = mean_coef - margin
    upper_bound_log = mean_coef + margin
    
    # Return aggregated log-scale metrics AND the exponentiated Hazard Ratios
    return pd.Series({
        'log_mean': mean_coef,
        'HR_mean': np.exp(mean_coef),
        'HR_lower_95CI': np.exp(lower_bound_log),
        'HR_upper_95CI': np.exp(upper_bound_log),
        'nb_corrected_se': corrected_se,
        'abs_log_mean': np.abs(mean_coef) # Used for sorting top 10
    })

# 2. Apply the correction to group by feature for each dataset
feature_subset_deepsurv_summary = feature_subset_deepsurv.groupby('feature').apply(calculate_nadeau_bengio_bounds).reset_index()
powershap_deepsurv_summary = powershap_deepsurv.groupby('feature').apply(calculate_nadeau_bengio_bounds).reset_index()
shap_filter_deepsurv_summary = shap_filter_deepsurv.groupby('feature').apply(calculate_nadeau_bengio_bounds).reset_index()

feature_subset_cox_summary = feature_subset_cox.groupby('feature').apply(calculate_nadeau_bengio_bounds).reset_index()
powershap_cox_summary = powershap_cox.groupby('feature').apply(calculate_nadeau_bengio_bounds).reset_index()
shap_filter_cox_summary = shap_filter_cox.groupby('feature').apply(calculate_nadeau_bengio_bounds).reset_index()

# 3. Sort by the absolute magnitude of the log-mean to get the top 10 features
feature_subset_deepsurv_summary_top = feature_subset_deepsurv_summary.sort_values(by="abs_log_mean", ascending=False).head(10).reset_index(drop=True)
powershap_deepsurv_summary_top = powershap_deepsurv_summary.sort_values(by="abs_log_mean", ascending=False).head(10).reset_index(drop=True)
shap_filter_deepsurv_summary_top = shap_filter_deepsurv_summary.sort_values(by="abs_log_mean", ascending=False).head(10).reset_index(drop=True)

feature_subset_cox_summary_top = feature_subset_cox_summary.sort_values(by="abs_log_mean", ascending=False).head(10).reset_index(drop=True)
powershap_cox_summary_top = powershap_cox_summary.sort_values(by="abs_log_mean", ascending=False).head(10).reset_index(drop=True)
shap_filter_cox_summary_top = shap_filter_cox_summary.sort_values(by="abs_log_mean", ascending=False).head(10).reset_index(drop=True)

# 4. Attach the human-readable variable labels used for the plot axes.
#    Any feature missing from var_labels keeps its raw name so nothing drops off a plot.
def label_summary(summary_top):
    labeled = add_labels(summary_top)
    labeled["label"] = labeled["label"].fillna(labeled["feature"])
    return labeled

feature_subset_deepsurv_summary_top = label_summary(feature_subset_deepsurv_summary_top)
powershap_deepsurv_summary_top = label_summary(powershap_deepsurv_summary_top)
shap_filter_deepsurv_summary_top = label_summary(shap_filter_deepsurv_summary_top)

feature_subset_cox_summary_top = label_summary(feature_subset_cox_summary_top)
powershap_cox_summary_top = label_summary(powershap_cox_summary_top)
shap_filter_cox_summary_top = label_summary(shap_filter_cox_summary_top)

all_summaries_top = [
    feature_subset_deepsurv_summary_top, powershap_deepsurv_summary_top, shap_filter_deepsurv_summary_top,
    feature_subset_cox_summary_top, powershap_cox_summary_top, shap_filter_cox_summary_top,
]

# Flag any top-10 feature that still needs an entry in var_labels
unlabeled = sorted({
    f
    for df in all_summaries_top
    for f in df.loc[~df["feature"].isin(var_labels["feature"]), "feature"]
})
if unlabeled:
    print("Features with no entry in var_labels (raw name shown on plots):")
    for f in unlabeled:
        print(f"  {f}")
    print()

# 5. Shared y-axis builder: ordered union of features across summaries, plus the matching tick labels
def feature_axis(*summaries):
    universe = (
        pd.concat([s[["feature", "label"]] for s in summaries])
        .drop_duplicates(subset="feature")
        .reset_index(drop=True)
    )
    feature_to_y = {feature: i for i, feature in enumerate(universe["feature"])}
    return feature_to_y, universe["label"].tolist()

# 6. Print the top 10 features for each method
columns_to_print = ['feature', 'label', 'HR_mean', 'HR_lower_95CI', 'HR_upper_95CI']

print("Feature Subset DeepSurv Top 10 Features:")
print(feature_subset_deepsurv_summary_top[columns_to_print])
print("\nPowerShap DeepSurv Top 10 Features:")
print(powershap_deepsurv_summary_top[columns_to_print])
print("\nSHAP Filter DeepSurv Top 10 Features:")
print(shap_filter_deepsurv_summary_top[columns_to_print])

print("\nFeature Subset Cox Top 10 Features:")
print(feature_subset_cox_summary_top[columns_to_print])
print("\nPowerShap Cox Top 10 Features:")
print(powershap_cox_summary_top[columns_to_print])
print("\nSHAP Filter Cox Top 10 Features:")
print(shap_filter_cox_summary_top[columns_to_print])

# Feature Subset

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 1. Isolate the complete universe of unique features across both top subsets,
#    along with the display label for each one
feature_to_y, all_labels = feature_axis(
    feature_subset_cox_summary_top,
    feature_subset_deepsurv_summary_top
)

# 2. Map the specific y-coordinates for each model based on its contained features
y_pos_cox = np.array([feature_to_y[f] for f in feature_subset_cox_summary_top["feature"]])
y_pos_deepsurv = np.array([feature_to_y[f] for f in feature_subset_deepsurv_summary_top["feature"]])

# Define the vertical displacement
offset = 0.2 

# Allocate sufficient vertical dimension for the combined feature set
plt.figure(figsize=(10, 12), layout="constrained") 

# Plot the Cox model data applying the negative offset to its specific coordinate array
plt.errorbar(
    feature_subset_cox_summary_top["HR_mean"], 
    y_pos_cox - offset, 
    xerr=[
        feature_subset_cox_summary_top["HR_mean"] - feature_subset_cox_summary_top["HR_lower_95CI"], 
        feature_subset_cox_summary_top["HR_upper_95CI"] - feature_subset_cox_summary_top["HR_mean"]
    ], 
    fmt="o", 
    label="Cox"
)

# Plot the DeepSurv model data applying the positive offset to its specific coordinate array
# Utilizing the risk_ratio columns specific to the DeepSurv dataframe
plt.errorbar(
    feature_subset_deepsurv_summary_top["HR_mean"], 
    y_pos_deepsurv + offset, 
    xerr=[
        feature_subset_deepsurv_summary_top["HR_mean"] - feature_subset_deepsurv_summary_top["HR_lower_95CI"], 
        feature_subset_deepsurv_summary_top["HR_upper_95CI"] - feature_subset_deepsurv_summary_top["HR_mean"]
    ], 
    fmt="o", 
    label="DeepSurv"
)

# Render the human-readable variable labels onto the base integer y-axis
plt.yticks(np.arange(len(all_labels)), all_labels)

plt.xlabel("Mean Relative Risk")
plt.ylabel("Feature")
plt.title("Relative Risk Values of Cox and DeepSurv Model\nFeatures Chosen by Science-Driven Feature Selection Ordered by Absolute Value")

plt.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), borderaxespad=0)
plt.axvline(1, color="gray", linestyle="--")

# Reverse the order of the y-axis
plt.gca().invert_yaxis()

plt.savefig(f"{OUTPUT_DIR}/feature_subset.png", dpi=300)

plt.show()

# PowerSHAP

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 1. Isolate the complete universe of unique features across both top subsets,
#    along with the display label for each one
feature_to_y, all_labels = feature_axis(
    powershap_cox_summary_top,
    powershap_deepsurv_summary_top
)

# 2. Map the specific y-coordinates for each model based on its contained features
y_pos_cox = np.array([feature_to_y[f] for f in powershap_cox_summary_top["feature"]])
y_pos_deepsurv = np.array([feature_to_y[f] for f in powershap_deepsurv_summary_top["feature"]])

# Define the vertical displacement
offset = 0.2 

# Allocate sufficient vertical dimension
plt.figure(figsize=(10, 12), layout="constrained") 

# Plot the Cox model data applying the negative offset to its specific coordinate array

plt.errorbar(
    powershap_cox_summary_top["HR_mean"], 
    y_pos_cox - offset, 
    xerr=[
        powershap_cox_summary_top["HR_mean"] - powershap_cox_summary_top["HR_lower_95CI"], 
        powershap_cox_summary_top["HR_upper_95CI"] - powershap_cox_summary_top["HR_mean"]
    ], 
    fmt="o", 
    label="Cox"
)

# Plot the DeepSurv model data applying the positive offset to its specific coordinate array
# Utilizing the risk_ratio columns specific to the DeepSurv dataframe
plt.errorbar(
    powershap_deepsurv_summary_top["HR_mean"], 
    y_pos_deepsurv + offset, 
    xerr=[
        powershap_deepsurv_summary_top["HR_mean"] - powershap_deepsurv_summary_top["HR_lower_95CI"], 
        powershap_deepsurv_summary_top["HR_upper_95CI"] - powershap_deepsurv_summary_top["HR_mean"]
    ], 
    fmt="o", 
    label="DeepSurv"
)

# Render the human-readable variable labels onto the base integer y-axis
plt.yticks(np.arange(len(all_labels)), all_labels)

plt.xlabel("Mean Relative Risk")
plt.ylabel("Feature")
plt.title("Relative Risk Values of Cox and DeepSurv Model\nFeatures Chosen by Powershap Ordered by Absolute Value")

plt.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), borderaxespad=0)
plt.axvline(1, color="gray", linestyle="--")

# Reverse the order of the y-axis
plt.gca().invert_yaxis()

plt.savefig(f"{OUTPUT_DIR}/powershap.png", dpi=300)

plt.show()

# SHAP Filter

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 1. Isolate the complete universe of unique features across both top subsets,
#    along with the display label for each one
feature_to_y, all_labels = feature_axis(
    shap_filter_cox_summary_top,
    shap_filter_deepsurv_summary_top
)

# 2. Map the specific y-coordinates for each model based on its contained features
y_pos_cox = np.array([feature_to_y[f] for f in shap_filter_cox_summary_top["feature"]])
y_pos_deepsurv = np.array([feature_to_y[f] for f in shap_filter_deepsurv_summary_top["feature"]])

# Define the vertical displacement
offset = 0.2 

# Allocate sufficient vertical dimension
plt.figure(figsize=(10, 12), layout="constrained") 

# Plot the Cox model data applying the negative offset to its specific coordinate array
plt.errorbar(
    shap_filter_cox_summary_top["HR_mean"], 
    y_pos_cox - offset, 
    xerr=[
        shap_filter_cox_summary_top["HR_mean"] - shap_filter_cox_summary_top["HR_lower_95CI"], 
        shap_filter_cox_summary_top["HR_upper_95CI"] - shap_filter_cox_summary_top["HR_mean"]
    ], 
    fmt="o", 
    label="Cox"
)

# Plot the DeepSurv model data applying the positive offset to its specific coordinate array
# Utilizing the risk_ratio columns specific to the DeepSurv dataframe
plt.errorbar(
    shap_filter_deepsurv_summary_top["HR_mean"], 
    y_pos_deepsurv + offset, 
    xerr=[
        shap_filter_deepsurv_summary_top["HR_mean"] - shap_filter_deepsurv_summary_top["HR_lower_95CI"], 
        shap_filter_deepsurv_summary_top["HR_upper_95CI"] - shap_filter_deepsurv_summary_top["HR_mean"]
    ], 
    fmt="o", 
    label="DeepSurv"
)

# Render the human-readable variable labels onto the base integer y-axis
plt.yticks(np.arange(len(all_labels)), all_labels)

plt.xlabel("Mean Relative Risk")
plt.ylabel("Feature")
plt.title("Relative Risk Values of Cox and DeepSurv Model\nFeatures Chosen by SHAP Filter Ordered by Absolute Value")

plt.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), borderaxespad=0)
plt.axvline(1, color="gray", linestyle="--")

# Reverse the order of the y-axis
plt.gca().invert_yaxis()

plt.savefig(f"{OUTPUT_DIR}/shap_filter.png", dpi=300)

plt.show()

# Cox

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 1. Isolate the complete universe of unique features across all three top subsets,
#    along with the display label for each one
feature_to_y, all_labels = feature_axis(
    feature_subset_cox_summary_top,
    powershap_cox_summary_top,
    shap_filter_cox_summary_top
)

# 2. Map the specific y-coordinates for each model based on its contained features
y_pos_subset = np.array([feature_to_y[f] for f in feature_subset_cox_summary_top["feature"]])
y_pos_power = np.array([feature_to_y[f] for f in powershap_cox_summary_top["feature"]])
y_pos_shap = np.array([feature_to_y[f] for f in shap_filter_cox_summary_top["feature"]])

# Define the vertical displacement
offset = 0.25 

# Expanded vertical dimension to mitigate overlapping error bars
plt.figure(figsize=(10, 12), layout="constrained") 

# Plot Feature Subset applying a negative offset to its specific coordinate array
plt.errorbar(
    feature_subset_cox_summary_top["HR_mean"], 
    y_pos_subset - offset, 
    xerr=[
        feature_subset_cox_summary_top["HR_mean"] - feature_subset_cox_summary_top["HR_lower_95CI"], 
        feature_subset_cox_summary_top["HR_upper_95CI"] - feature_subset_cox_summary_top["HR_mean"]
    ], 
    fmt="o", 
    label="Feature Subset"
)

# Plot PowerShap directly on the base coordinate (no shift)
plt.errorbar(
    powershap_cox_summary_top["HR_mean"], 
    y_pos_power, 
    xerr=[
        powershap_cox_summary_top["HR_mean"] - powershap_cox_summary_top["HR_lower_95CI"], 
        powershap_cox_summary_top["HR_upper_95CI"] - powershap_cox_summary_top["HR_mean"]
    ], 
    fmt="o", 
    label="PowerShap"
)

# Plot SHAP Filter applying a positive offset to its specific coordinate array
plt.errorbar(
    shap_filter_cox_summary_top["HR_mean"], 
    y_pos_shap + offset, 
    xerr=[
        shap_filter_cox_summary_top["HR_mean"] - shap_filter_cox_summary_top["HR_lower_95CI"], 
        shap_filter_cox_summary_top["HR_upper_95CI"] - shap_filter_cox_summary_top["HR_mean"]
    ], 
    fmt="o", 
    label="Shap Filter"
)

# Render the human-readable variable labels onto the base integer y-axis
plt.yticks(np.arange(len(all_labels)), all_labels)

# Standard formatting
plt.xlabel("Mean Relative Risk")
plt.ylabel("Feature")
plt.title("Relative Risk Values of Cox Model Features Ordered by Absolute Value")

plt.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), borderaxespad=0)
plt.axvline(1, color="gray", linestyle="--")

# Reverse the order of the y-axis
plt.gca().invert_yaxis()

plt.savefig(f"{OUTPUT_DIR}/cox.png", dpi=300)

plt.show()

# DeepSurv

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 1. Isolate the complete universe of unique features across all three DeepSurv models,
#    along with the display label for each one
feature_to_y, all_labels = feature_axis(
    feature_subset_deepsurv_summary_top,
    powershap_deepsurv_summary_top,
    shap_filter_deepsurv_summary_top
)

# 2. Map the specific y-coordinates for each model based on its contained features
y_pos_subset = np.array([feature_to_y[f] for f in feature_subset_deepsurv_summary_top["feature"]])
y_pos_power = np.array([feature_to_y[f] for f in powershap_deepsurv_summary_top["feature"]])
y_pos_shap = np.array([feature_to_y[f] for f in shap_filter_deepsurv_summary_top["feature"]])

# Define the vertical displacement
offset = 0.25 

# Expanded vertical dimension to mitigate overlapping error bars
plt.figure(figsize=(10, 12), layout="constrained") 

# Plot Feature Subset applying a negative offset to its specific coordinate array
plt.errorbar(
    feature_subset_deepsurv_summary_top["HR_mean"], 
    y_pos_subset - offset, 
    xerr=[
        feature_subset_deepsurv_summary_top["HR_mean"] - feature_subset_deepsurv_summary_top["HR_lower_95CI"], 
        feature_subset_deepsurv_summary_top["HR_upper_95CI"] - feature_subset_deepsurv_summary_top["HR_mean"]
    ], 
    fmt="o", 
    label="Feature Subset"
)

# Plot PowerShap directly on the base coordinate (no shift)
plt.errorbar(
    powershap_deepsurv_summary_top["HR_mean"], 
    y_pos_power, 
    xerr=[
        powershap_deepsurv_summary_top["HR_mean"] - powershap_deepsurv_summary_top["HR_lower_95CI"], 
        powershap_deepsurv_summary_top["HR_upper_95CI"] - powershap_deepsurv_summary_top["HR_mean"]
    ], 
    fmt="o", 
    label="PowerShap"
)

# Plot SHAP Filter applying a positive offset to its specific coordinate array
plt.errorbar(
    shap_filter_deepsurv_summary_top["HR_mean"], 
    y_pos_shap + offset, 
    xerr=[
        shap_filter_deepsurv_summary_top["HR_mean"] - shap_filter_deepsurv_summary_top["HR_lower_95CI"], 
        shap_filter_deepsurv_summary_top["HR_upper_95CI"] - shap_filter_deepsurv_summary_top["HR_mean"]
    ], 
    fmt="o", 
    label="Shap Filter"
)

# Render the human-readable variable labels onto the base integer y-axis
plt.yticks(np.arange(len(all_labels)), all_labels)

# Standard formatting
plt.xlabel("Mean Relative Risk")
plt.ylabel("Feature")
plt.title("Relative Risk Values of DeepSurv Model Features Ordered by Absolute Value")

plt.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), borderaxespad=0)
plt.axvline(1, color="gray", linestyle="--")

# Reverse the order of the y-axis
plt.gca().invert_yaxis()

plt.savefig(f"{OUTPUT_DIR}/deepsurv.png", dpi=300)

plt.show()